# Attention Extraction Demo
## Extract and visualize BERT attention without training

This notebook demonstrates:
1. Loading pre-trained BERT
2. Extracting attention weights for text
3. Visualizing attention heatmaps
4. Measuring head importance

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Use pre-trained BERT (no download needed if cached)
print("Loading BERT...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()
print("✓ BERT loaded!")

## Step 1: Encode Text and Extract Attention

In [ ]:
# Example sentiment text
text = "I love this movie! It's amazing and wonderful."

# Tokenize
inputs = tokenizer(text, return_tensors='pt', padding=True)
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

print(f"Text: {text}")
print(f"Tokens: {tokens}")
print(f"Token IDs shape: {inputs['input_ids'].shape}")

In [ ]:
# Extract attention
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

# Get attention weights
attentions = outputs.attentions  # Tuple of (12 layers)
# Each layer: (batch=1, num_heads=12, seq_len, seq_len)

print(f"Number of layers: {len(attentions)}")
print(f"Attention shape per layer: {attentions[0].shape}")
print(f"Total attention heads: {len(attentions) * attentions[0].shape[1]}")

## Step 2: Visualize Attention Heatmap

In [ ]:
def plot_attention(attention_matrix, tokens, layer, head, title=None):
    """Plot attention heatmap for one head"""
    plt.figure(figsize=(10, 8))
    
    sns.heatmap(
        attention_matrix,
        xticklabels=tokens,
        yticklabels=tokens,
        cmap='Blues',
        cbar=True,
        square=True,
        linewidths=0.5
    )
    
    plt.title(title or f"Attention Pattern: Layer {layer}, Head {head}")
    plt.xlabel("Key (Attended Token)")
    plt.ylabel("Query (Attending Token)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Visualize Layer 6, Head 3 (often shows interesting patterns)
layer_idx = 6
head_idx = 3
attention_matrix = attentions[layer_idx][0, head_idx].numpy()

plot_attention(attention_matrix, tokens, layer_idx, head_idx)

## Step 3: Analyze Multiple Heads

In [ ]:
# Visualize attention patterns from different layers
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Attention Patterns Across Layers and Heads", fontsize=16)

# Select interesting layers and heads
selections = [(0, 0), (3, 5), (6, 3), (9, 2), (11, 7), (11, 11)]

for idx, (layer, head) in enumerate(selections):
    ax = axes[idx // 3, idx % 3]
    attn = attentions[layer][0, head].numpy()
    
    im = ax.imshow(attn, cmap='Blues', aspect='auto')
    ax.set_title(f"Layer {layer}, Head {head}")
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Step 4: Measure Head Importance

In [ ]:
# Calculate attention flow: How much information flows through each head?
def calculate_attention_flow(attention_matrix):
    """Measure attention concentration (entropy-based)"""
    # Higher entropy = more dispersed attention
    # Lower entropy = more focused attention
    probs = attention_matrix + 1e-10  # Avoid log(0)
    entropy = -np.sum(probs * np.log(probs), axis=-1).mean()
    return entropy

# Calculate for all heads
head_importance = []
for layer_idx in range(12):
    for head_idx in range(12):
        attn = attentions[layer_idx][0, head_idx].numpy()
        flow = calculate_attention_flow(attn)
        head_importance.append({
            'layer': layer_idx,
            'head': head_idx,
            'entropy': flow
        })

# Sort by importance (lower entropy = more focused = more important)
head_importance_sorted = sorted(head_importance, key=lambda x: x['entropy'])

print("Top 10 Most Focused Attention Heads:")
for i, head in enumerate(head_importance_sorted[:10]):
    print(f"{i+1}. Layer {head['layer']}, Head {head['head']}: Entropy = {head['entropy']:.3f}")

## Step 5: Focus on Sentiment Words

In [ ]:
# Find which heads focus on sentiment words
sentiment_words = ['love', 'amazing', 'wonderful']
sentiment_indices = [i for i, token in enumerate(tokens) if token in sentiment_words]

print(f"Sentiment words found at indices: {sentiment_indices}")
print(f"Tokens: {[tokens[i] for i in sentiment_indices]}")

# Calculate attention to sentiment words for each head
sentiment_attention = []
for layer_idx in range(12):
    for head_idx in range(12):
        attn = attentions[layer_idx][0, head_idx].numpy()
        
        # Average attention TO sentiment words (from all other tokens)
        attn_to_sentiment = attn[:, sentiment_indices].mean()
        
        sentiment_attention.append({
            'layer': layer_idx,
            'head': head_idx,
            'sentiment_focus': attn_to_sentiment
        })

# Find heads that focus most on sentiment
sentiment_sorted = sorted(sentiment_attention, key=lambda x: x['sentiment_focus'], reverse=True)

print("\nTop 10 Heads Focusing on Sentiment Words:")
for i, head in enumerate(sentiment_sorted[:10]):
    print(f"{i+1}. Layer {head['layer']}, Head {head['head']}: Focus = {head['sentiment_focus']:.3f}")

## Next Steps

1. **Try different texts**: Change the input text to see how attention patterns differ
2. **Test multiple samples**: Load synthetic MOSEI data and analyze across samples
3. **Compare sentiment**: Analyze positive vs negative texts to find differences
4. **Implement ablation**: Zero out heads to measure impact on downstream task
5. **Build interpreter class**: Create `AttentionMechanisticInterpreter` based on these patterns